# Completed SSL classifier epoch extension

Stage-specific executable entry point using shared `src/` modules. This notebook contains no trained outputs and does not establish historical reproduction. See [execution order](README.md) and [stage guide](docs/reproduction.md).

## Methodology

Extend completed cache-backed classifier runs with ALL saved model, optimizer, scheduler, RNG and history state. Only the total epoch limit and new output location may change. Source best.pth and its epoch remain preserved if later epochs do not improve validation macro AUROC. The explicit hold_completed_lr policy retains the completed scheduler and terminal learning rate; it does not reset a cosine schedule.

## Required inputs

Both completed source task directories: last.pth, best.pth, config.json, summary.json, status.json and history.csv; plus every SSL/PTB-XL artifact referenced by their saved configs. Defaults identify the historical epoch-5 session in the verified arnavvt/checkpoint-2 input. For newly completed notebook03 runs, replace BOTH source directories with the saved full exports. Preserve required source-config mount paths; substantive provenance drift must reject.

## Key settings and execution

Only `MODE="full"` is supported; EXECUTE=False prevents any work until reviewed. The example extends epoch5 to epoch15. There is no new smoke optimization stage or new optimizer. An already completed epoch15 source requires a larger explicit target, not a restart. Incompatible settings/data fail closed; do not reinterpret a provenance failure as permission to initialize fresh weights.

On Kaggle enable GPU, Internet and the GITHUB_TOKEN secret. Review the configuration, set EXECUTE=True, then Run All. Secure synchronization and dependency setup run in fresh processes; no kernel restart is assumed. Settings are resolved from the synchronized repository snapshot and saved with its commit. See configs/kaggle_defaults.json for inherited settings.

In [ ]:
# Review this cell; nothing runs until EXECUTE=True.
EXECUTE = False
MODE = 'full'  # smoke/full for 01-03; full only for 04-05
GIT_BRANCH = 'refactor/project-organization'  # after merge, select the reviewed main branch explicitly
CHECKOUT_DIR = "/kaggle/working/BTP_project_files"
OUTPUT_ROOT = "/kaggle/working/btp_outputs"
OVERRIDES = {'SSL_FINETUNE_RESUME_RUN_DIRS': {'superclass': '/kaggle/input/datasets/arnavvt/checkpoint-2/btp_outputs/sessions/20261029T180106Z_e154ce34/ssl_finetuning/full/benchmark/mamba_superclass_20261029T180304Z_bb3fcff5',
                                  'subclass': '/kaggle/input/datasets/arnavvt/checkpoint-2/btp_outputs/sessions/20261029T180106Z_e154ce34/ssl_finetuning/full/benchmark/mamba_subclass_20261029T200603Z_991377cb'},
 'SSL_FINETUNE_RESUME_SCHEDULER_POLICY': 'hold_completed_lr',
 'SSL_FINETUNE_RESUME_TO_EPOCHS': 15}
REQUEST = dict(WORKFLOW_PROFILE='ssl_extended_finetuning', WORKFLOW_MODE=MODE,
    WORKFLOW_OVERRIDES=OVERRIDES, CHECKOUT_DIR=CHECKOUT_DIR,
    OUTPUT_ROOT=OUTPUT_ROOT, GIT_BRANCH=GIT_BRANCH, GIT_TIMEOUT_SECONDS=600)


In [ ]:
"""Self-contained standard-library bootstrap, embedded verbatim in the notebook.

The GitHub secret exists only in memory and Git child environments. It never
appears in URLs, argv, configuration, askpass file contents or returned results.
"""
import importlib
import json
import os
from pathlib import Path
import re
import signal
import subprocess
import sys
import tempfile
import time
import urllib.parse
from datetime import datetime, timezone
from uuid import uuid4


def _git(arguments, *, cwd=None, env=None, timeout=600, label="Git"):
    command = ["git", "-c", "credential.helper=", "-c", "core.hooksPath=" + os.devnull,
               "-c", "core.fsmonitor=false", *arguments]
    process = subprocess.Popen(command, cwd=cwd, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                               stderr=subprocess.PIPE, text=True, errors="replace")
    start = time.monotonic()
    try:
        while True:
            try:
                stdout, stderr = process.communicate(timeout=min(30, timeout))
                break
            except subprocess.TimeoutExpired:
                print(f"{label} still running ({time.monotonic() - start:.0f}s)", flush=True)
                if time.monotonic() - start >= timeout:
                    raise RuntimeError(f"{label} timed out; authentication details suppressed") from None
    except BaseException:
        if process.poll() is None:
            process.kill()
            process.communicate()
        raise
    if process.returncode:
        # Never forward authentication-related subprocess output or exception repr.
        raise RuntimeError(f"{label} failed (exit {process.returncode}). Check Internet, repository read permission, "
                           "branch and Kaggle GITHUB_TOKEN secret. Git output suppressed for credential safety.") from None
    return stdout.strip()


def kaggle_token():
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("GITHUB_TOKEN")
        if not isinstance(value, str) or not value.strip():
            raise ValueError("empty secret")
        return value.strip()
    except Exception:
        raise RuntimeError("Cannot retrieve Kaggle secret GITHUB_TOKEN. Add it in Kaggle Secrets and grant this notebook access.") from None


def sync_repository(checkout, *, branch="dev/unified-ecg-pipeline", token_provider=None,
                    repository_url="https://github.com/T-arn21/BTP_project_files.git", timeout=600):
    """Clone/update clean checkouts by fast-forward only; never stash/reset/clean."""
    checkout = Path(checkout).resolve()
    parsed = urllib.parse.urlsplit(repository_url)
    if parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError("Credential-bearing or parameterized Git URLs are forbidden")
    if parsed.scheme and parsed.scheme != "https" and not Path(repository_url).exists():
        raise ValueError("Only HTTPS repositories or explicit local test repositories are supported")
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]*", branch) or ".." in branch:
        raise ValueError("Invalid branch name")
    env = {k: v for k, v in os.environ.items()
           if not k.startswith(("GIT_TRACE", "GIT_CONFIG_")) and k not in {"GIT_CURL_VERBOSE", "GITHUB_TOKEN", "BTP_GITHUB_TOKEN"}}
    env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="never", GIT_CONFIG_NOSYSTEM="1", GIT_CONFIG_GLOBAL=os.devnull)
    exists = checkout.exists() and any(checkout.iterdir())
    if exists:
        top = _git(["rev-parse", "--show-toplevel"], cwd=checkout, env=env, label="Repository safety check")
        if Path(top).resolve() != checkout:
            raise RuntimeError("Checkout path is not its own Git repository; refusing to overwrite existing files")
        if _git(["status", "--porcelain", "--untracked-files=all"], cwd=checkout, env=env):
            raise RuntimeError("Uncommitted tracked/untracked local changes detected; checkout left untouched. Save them before unattended synchronization.")
        remote = _git(["remote", "get-url", "--all", "origin"], cwd=checkout, env=env)
        if remote != repository_url:
            raise RuntimeError("Existing origin differs from the configured repository; refusing synchronization")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    token = (token_provider or kaggle_token)()
    if not isinstance(token, str) or not token:
        raise RuntimeError("A nonempty GitHub credential is required")
    helper = None
    auth_env = None
    try:
        # File contains only environment lookup, never the secret itself.
        with tempfile.NamedTemporaryFile("w", suffix="_btp_askpass.py", delete=False, encoding="utf-8") as stream:
            helper = Path(stream.name)
            stream.write("#!/usr/bin/env python3\nimport os, sys\n"
                         "print('x-access-token' if 'username' in ' '.join(sys.argv[1:]).lower() "
                         "else os.environ['BTP_GITHUB_TOKEN'])\n")
        helper.chmod(0o700)
        auth_env = dict(env, GIT_ASKPASS=str(helper), BTP_GITHUB_TOKEN=token)
        if not exists:
            _git(["clone", "--single-branch", "--branch", branch, repository_url, str(checkout)],
                 env=auth_env, timeout=timeout, label="Git clone")
        else:
            _git(["fetch", "origin", f"+refs/heads/{branch}:refs/remotes/origin/{branch}"],
                 cwd=checkout, env=auth_env, timeout=timeout, label="Git fetch")
            if _git(["status", "--porcelain", "--untracked-files=all"], cwd=checkout, env=env):
                raise RuntimeError("Local changes appeared during fetch; refusing to update the checkout")
            # switch never forces files; Git refuses conflicting local state.
            _git(["switch", branch], cwd=checkout, env=env, label="Branch selection")
            _git(["merge-base", "--is-ancestor", "HEAD", f"origin/{branch}"], cwd=checkout, env=env,
                 label="Fast-forward safety check (local-only/divergent commits are preserved)")
            _git(["merge", "--ff-only", f"origin/{branch}"], cwd=checkout, env=env, label="Fast-forward update")
        head = _git(["rev-parse", "HEAD"], cwd=checkout, env=env)
        remote_head = _git(["rev-parse", f"origin/{branch}"], cwd=checkout, env=env)
        if head != remote_head or not re.fullmatch(r"[a-f0-9]{40,64}", head):
            raise RuntimeError("Synchronized checkout does not match the fetched remote branch")
        return {"checkout": str(checkout), "branch": branch, "commit": head}
    finally:
        if auth_env is not None:
            auth_env.pop("BTP_GITHUB_TOKEN", None)
        token = None
        if helper is not None:
            helper.unlink(missing_ok=True)


def select_import_root(checkout):
    checkout = Path(checkout).resolve()
    for name in list(sys.modules):
        if name == "src" or name.startswith("src."):
            del sys.modules[name]
    sys.path.insert(0, str(checkout))
    importlib.invalidate_caches()
    module = importlib.import_module("src")
    if Path(module.__file__).resolve().parent != checkout / "src":
        raise RuntimeError("Python resolved src outside the synchronized repository")
    return str(Path(module.__file__).resolve())


def launch_pipeline(checkout, config_path, session_dir):
    """Fresh controller/worker interpreters avoid kernel restarts and stale binaries."""
    checkout = Path(checkout).resolve()
    env = {k: v for k, v in os.environ.items() if k not in {"GITHUB_TOKEN", "BTP_GITHUB_TOKEN"}}
    env["PYTHONPATH"] = str(checkout)
    env["PYTHONUNBUFFERED"] = "1"
    env["PIP_NO_INPUT"] = "1"
    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    env["GIT_TERMINAL_PROMPT"] = "0"
    log = Path(session_dir) / "launcher.log"
    # No secret is available to the installer, worker or its logs.
    command = [sys.executable, "-u", "-m", "src.unattended", "--config", str(config_path)]
    process = subprocess.Popen(command, cwd=checkout, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, errors="replace",
                               start_new_session=os.name != "nt")
    import queue
    import threading
    lines = queue.Queue()
    def read():
        for line in process.stdout:
            lines.put(line)
        lines.put(None)
    threading.Thread(target=read, daemon=True).start()
    try:
        with log.open("a", encoding="utf-8", buffering=1) as stream:
            while True:
                try:
                    line = lines.get(timeout=30)
                except queue.Empty:
                    line = "Pipeline process is still running; see current stage in pipeline_status.json\n"
                if line is None:
                    break
                print(line, end="", flush=True)
                stream.write(line)
        return process.wait()
    except BaseException:
        if process.poll() is None:
            if os.name != "nt":
                os.killpg(process.pid, signal.SIGTERM)
            else:
                process.terminate()
        raise


def execute_notebook(settings):
    """Entry point for the notebook: always print a concise safe execution summary."""
    if any("TOKEN" in key.upper() or "SECRET" in key.upper() for key in settings):
        raise ValueError("Secrets must not be passed in execution settings; use Kaggle Secrets only")
    out = Path(settings["OUTPUT_ROOT"]).resolve()
    session = out / "sessions" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
    session.mkdir(parents=True, exist_ok=True)
    config = dict(settings, SESSION_DIR=str(session))
    summary_path = session / "final_summary.json"
    summary = {"status": "failed", "stage": "GitHub synchronization", "completed": {},
               "checks": {}, "errors": {}, "session_dir": str(session)}
    try:
        synchronized = sync_repository(settings["CHECKOUT_DIR"], branch=settings["GIT_BRANCH"],
                                       timeout=settings["GIT_TIMEOUT_SECONDS"])
        import_root = select_import_root(synchronized["checkout"])
        if "WORKFLOW_PROFILE" in settings:
            # Import only stdlib profile selection after secure synchronization;
            # scientific imports remain in fresh post-install worker processes.
            profiles = importlib.import_module("src.notebook_workflows")
            settings = profiles.workflow_settings(settings)
            config = dict(settings, SESSION_DIR=str(session))
        config["SOURCE_REVISION"] = synchronized["commit"]
        config["IMPORT_ROOT"] = import_root
        print(f"Synchronized {synchronized['branch']} @ {synchronized['commit'][:12]}; imports: {config['IMPORT_ROOT']}", flush=True)
        path = session / "execution_config.json"
        path.write_text(json.dumps(config, indent=2), encoding="utf-8")
        summary["checks"]["repository_sync"] = "passed"
        summary["stage"] = "dependency setup / pipeline worker"
        code = launch_pipeline(synchronized["checkout"], path, session)
        if summary_path.is_file():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
        if code != 0:
            summary["status"] = "failed"
            summary["errors"].setdefault("pipeline", "Pipeline process failed; inspect stage status and logs")
        return summary
    except BaseException as exc:
        # All synchronization exceptions are credential-free by construction.
        if summary_path.is_file():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
        summary["status"] = "failed"
        summary["errors"][summary.get("stage", "launcher")] = str(exc) or type(exc).__name__
        return summary
    finally:
        summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        print("EXECUTION SUMMARY:", json.dumps({"status": summary["status"], "checks": summary.get("checks", {}),
              "completed_experiments": list(summary.get("completed", {})), "errors": summary.get("errors", {}),
              "results": str(session)}, indent=2), flush=True)


if EXECUTE:
    EXECUTION_SUMMARY = execute_notebook(REQUEST)
    if EXECUTION_SUMMARY["status"] != "complete":
        raise RuntimeError("Workflow failed; inspect the printed summary and preserved session logs")
else:
    EXECUTION_SUMMARY = dict(status="not_executed", completed={}, session_dir=None)
    print("Configuration review only. Set EXECUTE=True to run this stage; no secret, network, installation or GPU used.")


## Outputs and interpretation

All session files are below OUTPUT_ROOT/sessions/<new_session>/. `ssl_finetuning/resumed/benchmark/mamba_<task>_<id>/`: source snapshot, inherited history plus new epochs, preserved prior best or a newly validation-selected best, checkpoints, configuration and complete evaluation exports.

Inspect final_summary.json and task status.json before reporting completion. Smoke/diagnostic outputs are not full benchmarks. Inputs and prior outputs are never overwritten.

In [ ]:
print("Status:", EXECUTION_SUMMARY["status"])
print("Completed:", list(EXECUTION_SUMMARY["completed"]))
print("Session:", EXECUTION_SUMMARY["session_dir"])